# Fine-tune Qwen2.5-3B-Instruct for Daemon Desktop Pet (SFT)

This notebook fine-tunes `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` using Supervised Fine-Tuning (SFT) with Unsloth on a free Colab T4 GPU.

**Dataset:** Alpaca-format JSONL with instruction/input/output columns

**Setup:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Click **Connect**
3. Run all cells in order

**Based on:** [unsloth-buddy](https://github.com/TYH-labs/unsloth-buddy) skill

## Cell 1: Install Unsloth & Dependencies

In [1]:
%%capture
!pip install unsloth
# Restart runtime if prompted (Runtime → Restart runtime)

## Cell 2: Verify GPU & Imports

In [ ]:
import torch, json
assert torch.cuda.is_available(), "No GPU detected! Go to Runtime → Change runtime type → T4 GPU"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

from unsloth import FastLanguageModel
import unsloth, trl, transformers, datasets

print(json.dumps({
    "gpu": gpu_name,
    "vram_gb": round(vram_gb, 1),
    "unsloth": unsloth.__version__,
    "trl": trl.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "cuda": torch.version.cuda,
}))
print("GPU ready!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


## Cell 3: Load Model & Apply LoRA

Loads `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` in 4-bit QLoRA (~2GB VRAM).

In [ ]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print(f"Model loaded. Trainable params: {model.print_trainable_parameters()}")

## Cell 4: Prepare Dataset

**Upload your dataset file** (batch_00000_alpaca_clean.jsonl), then run the cell below.

Expected format: Alpaca JSONL with `instruction`, `input`, `output` columns.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Upload your Alpaca-format JSONL dataset
# Supports instruction/input/output columns (Daemon dataset format)
# ══════════════════════════════════════════════════════════════════════════════

from google.colab import files
import json
from datasets import Dataset

# Upload your JSONL file
print('Upload batch_00000_alpaca_clean.jsonl:')
uploaded = files.upload()

# Load the uploaded file
filename = list(uploaded.keys())[0]
with open(filename, 'r') as f:
    data = [json.loads(line) for line in f.readlines()]

dataset = Dataset.from_list(data)

# Format to Qwen2.5 chat template for SFT
def format_chat(example):
    instruction = example.get('instruction', '')
    output = example.get('output', '')
    if not instruction or not output:
        return None
    messages = [
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": output},
    ]
    return {"messages": messages}

dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
dataset = dataset.filter(lambda x: x is not None and x.get('messages') is not None)

print(f'Dataset: {len(dataset)} samples')
print(f'Example: {dataset[0]}')

## Cell 5: Train with SFT

In [ ]:
from trl import SFTTrainer, SFTConfig

# Format Alpaca JSONL to Qwen2.5 chat template for SFT
def format_chat(example):
    instruction = example.get("instruction", "")
    input_text = example.get("input", "")
    output = example.get("output", "")
    if not instruction or not output:
        return None
    user_content = instruction
    if input_text:
        user_content += f"\n{input_text}"
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output},
    ]
    return {"messages": messages}

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    formatting_func = format_chat,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # Effective batch size = 8
        max_steps = 300,                   # Increase for real training (e.g., 300-500)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        warmup_steps = 10,
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"Training complete! Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## Cell 6: Save LoRA Adapters

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("Adapters saved to lora_model/")

# List saved files
import os
for f in os.listdir("lora_model"):
    size = os.path.getsize(os.path.join("lora_model", f))
    print(f"  {f}: {size / 1e6:.1f} MB")

## Cell 7: Test Inference

Test with a Daemon-style prompt to verify the model learned the personality.

In [ ]:
from unsloth import FastLanguageModel

# Reload for inference
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# Test with a Daemon-style prompt
messages = [
    {"role": "user", "content": "Mode: desktop_companion\nAPM: 120\nIdle: 30s\nWindow: vscode\nScreen: VS Code editor with Python source\nBrowser: https://github.com/\nMemory: user_habits: Uses AI for 90% of tasks | user_current_project: Daemon desktop pet\nTrigger: autonomous"},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, top_p=0.9)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

## Cell 8 (Optional): Export to GGUF

Export to GGUF format for use with Ollama, LM Studio, or llama.cpp.

In [ ]:
Uncomment to export:
model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")
print("GGUF exported!")

Download the GGUF file:
from google.colab import files
import glob
for f in glob.glob("model_gguf/*.gguf"):
    files.download(f)

## Cell 9 (Optional): Push to Hugging Face Hub

In [ ]:
Uncomment and set your HF token:
HF_TOKEN = "hf_YOUR_TOKEN_HERE"
model.push_to_hub("your-username/qwen2.5-3b-finetuned", token=HF_TOKEN)
tokenizer.push_to_hub("your-username/qwen2.5-3b-finetuned", token=HF_TOKEN)

## Download Adapters

Download the `lora_model/` folder from the Colab file browser (left panel → folder icon → right-click → Download).